In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["Pinecone_API_KEY"] = os.getenv("Pinecone_API_KEY")

In [11]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

.../ipykernel_3292\1338964493.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import PineconeHybridSearchRetriever


Create Pinecone index

In [12]:
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-langchain-pinecone"
api_key = os.getenv("Pinecone_API_KEY")

# initialize the Pinecone client
pc = Pinecone(api_key=api_key)

#u can also do this steps manually in the Pinecone console
# create the index (only if it does not exist yet)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # dimension of the dense vector (all-MiniLM-L6-v2 gives 384 numbers)
        metric="dotproduct",  # sparse values are supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)
index

Setup Both Searching Methods(Symentic and Syntatic search)

In [ ]:
#loads the embedding model.
#responsible for dense/Similarity Search we need embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2108.04it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
#responsible for Sparse/keyword search
#This creates the keyword encoder
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [ ]:
#train BM25 on your own sentences
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

## tfidf values on these sentences
bm25_encoder.fit(sentences)

## saves what it learned to json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")


100%|██████████| 3/3 [00:00<00:00, 1897.02it/s]


In [16]:
# connect to the Pinecone index we created above
index = pc.Index(index_name)


In [ ]:
#sets up the retriever.
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder, index=index)


In [ ]:
#For each sentence, the retriever makes a dense vector and a sparse vector and uploads both, with the text, to Pinecone. 
retriever.add_texts(
    [
        "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",
    ]
)


100%|██████████| 1/1 [00:02<00:00,  2.01s/it]


In [ ]:
#For the question, the retriever makes a dense and a sparse vector, asks Pinecone for the closest matches, and returns them as Document objects.
retriever.invoke("which city i visited  ") #meaning match 

[Document(metadata={'score': 0.278969347}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.26786682}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.260894358}, page_content='In 2022, I visited New York')]

In [ ]:
retriever.invoke("which city i visited in 2021 ") #keyword match

[Document(metadata={'score': 0.462777674}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.310391963}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.30464536}, page_content='In 2023, I visited Paris')]